# ML Baselines: 3-Fold Cross-Validation

This notebook runs ML baselines (XGBoost, Ridge) using the **same 3-fold CV splits** as the DL models for fair comparison.

Features: `cat(drug_embed, cellline_one_hot, rna_embed)` = 768 + num_celllines + 256 dimensions

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path
import json

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy import stats

try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    print("XGBoost not installed, will install...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'xgboost'])
    import xgboost as xgb

from gastro_transformer.data import (
    DrugEmbeddingDataset,
    IC50Dataset,
)

In [ ]:
# Configuration - must match DL CV setup
N_FOLDS = 3
SEED = 42

ROOT_DIR = '../'

# Data paths
DRUG_EMBEDDINGS_CSV = ROOT_DIR + 'data/drug_embeddings.csv'
IC50_CSV = ROOT_DIR + 'data/ic50_data.csv'
CELLLINE_RNA_CSV = ROOT_DIR + 'data/processed/ccle_rna_for_ic50.csv'

OUTPUT_DIR = Path(ROOT_DIR + 'reports/cross_validation_v4')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"N folds: {N_FOLDS}")
print(f"Seed: {SEED}")

In [ ]:
# Load data
print("Loading drug embeddings...")
drug_embeddings = DrugEmbeddingDataset(
    csv_path=DRUG_EMBEDDINGS_CSV,
    drug_dim=768
)

print("\nLoading IC50 dataset...")
ic50_dataset = IC50Dataset(
    ic50_csv_path=IC50_CSV,
    drug_embeddings=drug_embeddings,
    rna_csv_path=CELLLINE_RNA_CSV,
    rna_dim=256
)

print(f"\nTotal IC50 samples: {len(ic50_dataset)}")
print(f"Unique cell-lines: {ic50_dataset.num_celllines}")

In [ ]:
# Extract features for all samples
print("Extracting features...")

n_samples = len(ic50_dataset)
drug_dim = 768
rna_dim = 256
num_celllines = ic50_dataset.num_celllines

features = np.zeros((n_samples, drug_dim + num_celllines + rna_dim), dtype=np.float32)
targets = np.zeros(n_samples, dtype=np.float32)

for i in range(n_samples):
    item = ic50_dataset[i]
    
    # Drug embedding: 768 dims
    drug_embed = item['drug_embed'].numpy()
    features[i, :drug_dim] = drug_embed
    
    # Cell-line one-hot: num_celllines dims
    cl_idx = item['cellline_id'].item()
    features[i, drug_dim + cl_idx] = 1.0
    
    # RNA embedding: 256 dims
    if 'rna_embed' in item:
        rna_embed = item['rna_embed'].numpy()
        features[i, drug_dim + num_celllines:] = rna_embed
    
    # Target
    targets[i] = item['ic50'].item()

print(f"Feature matrix shape: {features.shape}")
print(f"Target shape: {targets.shape}")
print(f"Features per sample: {drug_dim} (drug) + {num_celllines} (cellline_onehot) + {rna_dim} (RNA) = {drug_dim + num_celllines + rna_dim}")

In [ ]:
# Create cell-line aware CV splits (same logic as DL models)
# Get unique cell-lines and their indices
print("Creating cell-line aware CV splits...")

rng = np.random.default_rng(SEED)

# Get unique cell-lines from dataset
unique_celllines = sorted(ic50_dataset.cellline_to_idx.keys())
rng.shuffle(unique_celllines)

# Split into N folds
n_celllines = len(unique_celllines)
fold_size = n_celllines // N_FOLDS

cellline_folds = []
for fold_idx in range(N_FOLDS):
    start = fold_idx * fold_size
    end = start + fold_size if fold_idx < N_FOLDS - 1 else n_celllines
    fold_celllines = set(unique_celllines[start:end])
    cellline_folds.append(fold_celllines)

print(f"Created {N_FOLDS} folds with {fold_size}-{fold_size+1} cell-lines each")

# Create sample-level fold assignments
sample_fold = np.zeros(n_samples, dtype=int)
for i in range(n_samples):
    cl_id = ic50_dataset.cellline_ids[i]
    for fold_idx, fold_cl in enumerate(cellline_folds):
        if cl_id in fold_cl:
            sample_fold[i] = fold_idx
            break

print(f"Sample distribution per fold: {np.bincount(sample_fold)}")

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute regression metrics."""
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    
    # Correlation metrics
    pearson_r, _ = stats.pearsonr(y_true, y_pred)
    spearman_r, _ = stats.spearmanr(y_true, y_pred)
    
    # R-squared
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    return {
        'mse': float(mse),
        'rmse': float(rmse),
        'mae': float(mae),
        'pearson_r': float(pearson_r),
        'spearman_r': float(spearman_r),
        'r2': float(r2)
    }

In [ ]:
# Train and evaluate ML models with CV
print("=" * 60)
print("Running ML Baselines with 3-Fold CV")
print("=" * 60)

results = {}

for model_name in ['xgboost', 'ridge']:
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print(f"{'='*60}")
    
    fold_metrics = []
    
    for fold_idx in range(N_FOLDS):
        print(f"\n--- Fold {fold_idx + 1}/{N_FOLDS} ---")
        
        # Train on remaining folds, test on this fold
        train_mask = sample_fold != fold_idx
        test_mask = sample_fold == fold_idx
        
        X_train = features[train_mask]
        y_train = targets[train_mask]
        X_test = features[test_mask]
        y_test = targets[test_mask]
        
        print(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")
        
        # Standardize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        if model_name == 'xgboost':
            # XGBoost with similar hyperparameters to benchmark
            model = xgb.XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=SEED,
                n_jobs=-1,
                verbosity=0
            )
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
        else:  # ridge
            model = Ridge(alpha=1.0, random_state=SEED)
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
        
        # Compute metrics
        metrics = compute_metrics(y_test, y_pred)
        fold_metrics.append(metrics)
        
        print(f"Fold {fold_idx + 1} - R²: {metrics['r2']:.4f}, Pearson: {metrics['pearson_r']:.4f}, Spearman: {metrics['spearman_r']:.4f}")
    
    # Aggregate metrics across folds
    avg_metrics = {}
    for key in fold_metrics[0].keys():
        values = [m[key] for m in fold_metrics]
        avg_metrics[key] = float(np.mean(values))
        avg_metrics[f'{key}_std'] = float(np.std(values))
    
    # Compute 95% CI
    for key in ['r2', 'pearson_r', 'spearman_r', 'rmse', 'mae']:
        values = [m[key] for m in fold_metrics]
        sem = np.std(values) / np.sqrt(N_FOLDS)
        avg_metrics[f'{key}_ci95'] = float(1.96 * sem)
    
    results[model_name] = {
        'fold_metrics': fold_metrics,
        'average_metrics': avg_metrics
    }
    
    print(f"\n{model_name.upper()} Average:")
    print(f"  R²: {avg_metrics['r2']:.4f} ± {avg_metrics['r2_std']:.4f}")
    print(f"  Pearson R: {avg_metrics['pearson_r']:.4f} ± {avg_metrics['pearson_r_std']:.4f}")
    print(f"  Spearman R: {avg_metrics['spearman_r']:.4f} ± {avg_metrics['spearman_r_std']:.4f}")

In [ ]:
# Add config to results
results['config'] = {
    'n_folds': N_FOLDS,
    'seed': SEED,
    'feature_dims': {
        'drug_embedding': drug_dim,
        'cellline_onehot': num_celllines,
        'rna_embedding': rna_dim,
        'total': drug_dim + num_celllines + rna_dim
    },
    'description': 'ML baselines using cell-line-aware 3-fold CV, same splits as DL models'
}

# Save results
output_path = OUTPUT_DIR / 'ml_baselines_cv_results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {output_path}")

In [ ]:
# Load existing DL CV results for comparison
dl_results_path = OUTPUT_DIR / 'dl_baselines_results.json'
with open(dl_results_path, 'r') as f:
    dl_results = json.load(f)

print("Loaded DL CV results from:", dl_results_path)

In [ ]:
# Create comparison table
print("\n" + "=" * 80)
print("COMPARISON: ML vs DL Baselines (3-Fold CV)")
print("=" * 80)

# Combine all results
all_results = {}

# Add ML results
for model_name in ['xgboost', 'ridge']:
    all_results[model_name] = results[model_name]['average_metrics']

# Add DL results
for model_name in ['detached_mlp', 'simple_mlp', 'qformer_v3', 'concat_v3', 'qformer_v4']:
    if model_name in dl_results:
        all_results[model_name] = dl_results[model_name]['average_metrics']

# Sort by R²
sorted_models = sorted(all_results.keys(), key=lambda x: all_results[x]['r2'], reverse=True)

# Print table
print(f"\n{'Model':<20} {'R²':>8} {'Pearson':>10} {'Spearman':>10} {'RMSE':>8} {'MAE':>8}")
print("-" * 64)
for model in sorted_models:
    m = all_results[model]
    print(f"{model:<20} {m['r2']:>8.4f} {m['pearson_r']:>10.4f} {m['spearman_r']:>10.4f} {m['rmse']:>8.4f} {m['mae']:>8.4f}")

In [ ]:
# Save combined comparison
combined_results = {
    'ml_baselines': results,
    'dl_baselines': dl_results,
    'comparison': {model: all_results[model] for model in sorted_models}
}

combined_path = OUTPUT_DIR / 'full_cv_comparison.json'
with open(combined_path, 'w') as f:
    json.dump(combined_results, f, indent=2)

print(f"Combined results saved to: {combined_path}")

In [ ]:
# Key findings
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

best_ml = max(['xgboost', 'ridge'], key=lambda x: results[x]['average_metrics']['r2'])
best_dl = max(['qformer_v4', 'concat_v3', 'qformer_v3', 'detached_mlp'], 
              key=lambda x: dl_results[x]['average_metrics']['r2'])

ml_r2 = results[best_ml]['average_metrics']['r2']
dl_r2 = dl_results[best_dl]['average_metrics']['r2']

print(f"\nBest ML model ({best_ml}): R² = {ml_r2:.4f}")
print(f"Best DL model ({best_dl}): R² = {dl_r2:.4f}")
print(f"DL advantage: +{(dl_r2 - ml_r2):.4f} ({((dl_r2 - ml_r2) / ml_r2 * 100):.1f}% relative)")

print("\nNote: ML baselines use cell-line one-hot encoding ({num_celllines} dims)")
print("This is more expressive than the learnable embeddings used in DL models,")
print("but still cannot capture drug-cellline interaction effects.")